# k-omega Spectra at Selected Spatial Slices

Pick a specific x or y location in the simulation and compute the k-omega spectrum for that 1D spatial slice over time.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from reconn_wave_power.io import read_simulation
from reconn_wave_power.spectrum import compute_komega_2d

%matplotlib inline

## Load Data

In [ ]:
INPUT_FILE = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/test_sim_data/test00/input/input"
OUTPUT_FOLDER = "/Users/colby/Research/Projects/My_Projects/2026.Reconn_Wave_Power/test_sim_data/test00/Output"

ds = read_simulation(
    input_file=INPUT_FILE,
    output_folder=OUTPUT_FOLDER,
    fields=("B",),
    progress=True,
)
ds

In [ ]:
dx = float(ds.x[1] - ds.x[0])
dy = float(ds.y[1] - ds.y[0])
dt = ds.attrs.get("dt", float(ds.time[1] - ds.time[0]))

print(f"dx = {dx}, dy = {dy}, dt = {dt}")
print(f"x range: [{float(ds.x.min())}, {float(ds.x.max())}]")
print(f"y range: [{float(ds.y.min())}, {float(ds.y.max())}]")

## Configuration

Choose the field component, slice direction, and location(s).

In [ ]:
FIELD = "Bz"

# Slice along a fixed y to get k_x vs omega
# Set to a list of y-values to compare multiple slices
Y_LOCATIONS = [50.0, 100.0, 150.0]  # in simulation units (d_i)

# Slice along a fixed x to get k_y vs omega
X_LOCATIONS = [200.0, 400.0, 600.0]  # in simulation units (d_i)

## k_x - omega at fixed y locations

In [ ]:
fig, axes = plt.subplots(1, len(Y_LOCATIONS), figsize=(6 * len(Y_LOCATIONS), 6), sharey=True)
if len(Y_LOCATIONS) == 1:
    axes = [axes]

for ax, y_loc in zip(axes, Y_LOCATIONS):
    # Select the slice: result is (time, x)
    da_slice = ds[FIELD].sel(y=y_loc, method="nearest")
    actual_y = float(da_slice.y)
    data_2d = da_slice.values  # (time, x)

    k, omega, P = compute_komega_2d(data_2d, dx=dx, dt=dt, axes=(1, 0))

    im = ax.pcolormesh(k, omega, P.T, shading="auto", cmap="inferno",
                       norm=plt.matplotlib.colors.LogNorm(vmin=P.max() * 1e-6, vmax=P.max()))
    ax.set_xlabel(r"$k_x$ [rad/$d_i$]")
    ax.set_title(f"y = {actual_y:.1f} $d_i$")
    ax.axhline(0, color="gray", lw=0.5, alpha=0.5)
    ax.axvline(0, color="gray", lw=0.5, alpha=0.5)

axes[0].set_ylabel(r"$\omega$ [rad/$\Omega_{ci}$]")
fig.colorbar(im, ax=axes, label="Power", shrink=0.8)
fig.suptitle(f"$k_x$-$\\omega$ spectra of {FIELD} at fixed y", fontsize=14)
plt.tight_layout()
plt.show()

## k_y - omega at fixed x locations

In [ ]:
fig, axes = plt.subplots(1, len(X_LOCATIONS), figsize=(6 * len(X_LOCATIONS), 6), sharey=True)
if len(X_LOCATIONS) == 1:
    axes = [axes]

for ax, x_loc in zip(axes, X_LOCATIONS):
    # Select the slice: result is (time, y)
    da_slice = ds[FIELD].sel(x=x_loc, method="nearest")
    actual_x = float(da_slice.x)
    data_2d = da_slice.values  # (time, y)

    k, omega, P = compute_komega_2d(data_2d, dx=dy, dt=dt, axes=(1, 0))

    im = ax.pcolormesh(k, omega, P.T, shading="auto", cmap="inferno",
                       norm=plt.matplotlib.colors.LogNorm(vmin=P.max() * 1e-6, vmax=P.max()))
    ax.set_xlabel(r"$k_y$ [rad/$d_i$]")
    ax.set_title(f"x = {actual_x:.1f} $d_i$")
    ax.axhline(0, color="gray", lw=0.5, alpha=0.5)
    ax.axvline(0, color="gray", lw=0.5, alpha=0.5)

axes[0].set_ylabel(r"$\omega$ [rad/$\Omega_{ci}$]")
fig.colorbar(im, ax=axes, label="Power", shrink=0.8)
fig.suptitle(f"$k_y$-$\\omega$ spectra of {FIELD} at fixed x", fontsize=14)
plt.tight_layout()
plt.show()

## Zoomed View

Re-plot with restricted k and omega ranges to focus on the low-frequency physics.

In [ ]:
K_MAX = 3.0    # rad/d_i
W_MAX = 3.0    # rad/Omega_ci

# Pick a single slice to zoom into
y_loc = Y_LOCATIONS[len(Y_LOCATIONS) // 2]
da_slice = ds[FIELD].sel(y=y_loc, method="nearest")
data_2d = da_slice.values

k, omega, P = compute_komega_2d(data_2d, dx=dx, dt=dt, axes=(1, 0))

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.pcolormesh(k, omega, P.T, shading="auto", cmap="inferno",
                   norm=plt.matplotlib.colors.LogNorm(vmin=P.max() * 1e-6, vmax=P.max()))
plt.colorbar(im, ax=ax, label="Power")
ax.set_xlabel(r"$k_x$ [rad/$d_i$]", fontsize=12)
ax.set_ylabel(r"$\omega$ [rad/$\Omega_{ci}$]", fontsize=12)
ax.set_title(f"$k_x$-$\\omega$ (zoomed) — {FIELD} at y = {float(da_slice.y):.1f} $d_i$", fontsize=14)
ax.set_xlim(-K_MAX, K_MAX)
ax.set_ylim(-W_MAX, W_MAX)
ax.axhline(0, color="gray", lw=0.5, alpha=0.5)
ax.axvline(0, color="gray", lw=0.5, alpha=0.5)
plt.tight_layout()
plt.show()

## Single Slice Summary

Set a y location and field below to see three views:
1. 2D snapshot of the field in real space (x vs y)
2. The time evolution of the slice at fixed y (x vs time)
3. The k-omega spectrum for that slice

In [ ]:
# --- Settings ---
SLICE_FIELD = "Bz"
SLICE_Y = 40.0        # y location in d_i
SNAPSHOT_TIME = 250     # timestep index for the 2D real-space plot
X_RANGE = (400, 800)          # Set to (xmin, xmax) to restrict x range, e.g. (200.0, 600.0). None for full domain.
T_RANGE = (100,750)          # Set to (tmin, tmax) to restrict time range, e.g. (500.0, 1500.0). None for full range.

# --- Get the data ---
da = ds[SLICE_FIELD]
if X_RANGE is not None:
    da = da.sel(x=slice(X_RANGE[0], X_RANGE[1]))
if T_RANGE is not None:
    da = da.sel(time=slice(T_RANGE[0], T_RANGE[1]))

da_snapshot = da.isel(time=SNAPSHOT_TIME)
da_slice = da.sel(y=SLICE_Y, method="nearest")
actual_y = float(da_slice.y)
dx_local = float(da.x[1] - da.x[0])
dt_local = float(da.time[1] - da.time[0])

fig, axes = plt.subplots(3, 1, figsize=(12, 18))

# Panel 1: 2D real-space snapshot (x vs y)
ax = axes[0]
snapshot_vals = da_snapshot.values
im0 = ax.pcolormesh(da.x, ds.y, snapshot_vals.T, shading="auto", cmap="RdBu_r")
ax.axhline(actual_y, color="lime", lw=1.5, ls="--", label=f"y = {actual_y:.1f}")
ax.set_xlabel(r"x [$d_i$]")
ax.set_ylabel(r"y [$d_i$]")
ax.set_title(f"{SLICE_FIELD} at t = {float(da_snapshot.time):.1f} $\\Omega_{{ci}}^{{-1}}$")
ax.set_aspect("equal")
ax.legend(loc="upper right")
plt.colorbar(im0, ax=ax, label=SLICE_FIELD)

# Panel 2: Time evolution of the slice (x vs time)
ax = axes[1]
slice_vals = da_slice.values  # (time, x)
im1 = ax.pcolormesh(da.x, da.time, slice_vals, shading="auto", cmap="RdBu_r")
ax.set_xlabel(r"x [$d_i$]")
ax.set_ylabel(r"time [$\Omega_{ci}^{-1}$]")
ax.set_title(f"{SLICE_FIELD}(x, t) at y = {actual_y:.1f} $d_i$")
plt.colorbar(im1, ax=ax, label=SLICE_FIELD)

# Panel 3: k-omega spectrum
ax = axes[2]
k, omega, P = compute_komega_2d(slice_vals, dx=dx_local, dt=dt_local, axes=(1, 0))
im2 = ax.pcolormesh(k, omega, P.T, shading="auto", cmap="inferno",
                    norm=plt.matplotlib.colors.LogNorm(vmin=P.max() * 1e-6, vmax=P.max()))
ax.set_xlabel(r"$k_x$ [rad/$d_i$]")
ax.set_ylabel(r"$\omega$ [rad/$\Omega_{ci}$]")
ax.set_title(f"$k_x$-$\\omega$ at y = {actual_y:.1f} $d_i$")
ax.axhline(0, color="gray", lw=0.5, alpha=0.5)
ax.axvline(0, color="gray", lw=0.5, alpha=0.5)
plt.colorbar(im2, ax=ax, label="Power")

x_label = f"x=[{float(da.x.min()):.0f}, {float(da.x.max()):.0f}]" if X_RANGE else "full x"
t_label = f"t=[{float(da.time.min()):.0f}, {float(da.time.max()):.0f}]" if T_RANGE else "full t"
fig.suptitle(f"Slice summary: {SLICE_FIELD} at y = {actual_y:.1f} $d_i$ ({x_label}, {t_label})", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import os
from matplotlib.animation import FuncAnimation

MOVIE_DIR = os.path.join(os.path.dirname(os.getcwd()), "movies") if os.path.basename(os.getcwd()) == "notebooks" else "movies"
os.makedirs(MOVIE_DIR, exist_ok=True)

# Compute fixed color limits across all timesteps
vmin = float(da.min())
vmax = float(da.max())
vlim = max(abs(vmin), abs(vmax))

fig, ax = plt.subplots(figsize=(12, 4))
snapshot_0 = da.isel(time=0).values
im = ax.pcolormesh(da.x, ds.y, snapshot_0.T, shading="auto", cmap="RdBu_r",
                   vmin=-vlim, vmax=vlim)
ax.axhline(actual_y, color="lime", lw=1.5, ls="--", label=f"y = {actual_y:.1f}")
ax.set_xlabel(r"x [$d_i$]")
ax.set_ylabel(r"y [$d_i$]")
ax.set_aspect("equal")
ax.legend(loc="upper right")
plt.colorbar(im, ax=ax, label=SLICE_FIELD)
title = ax.set_title("")

def update(frame):
    snapshot = da.isel(time=frame).values
    im.set_array(snapshot.T.ravel())
    title.set_text(f"{SLICE_FIELD} at t = {float(da.time[frame]):.1f} $\\Omega_{{ci}}^{{-1}}$")
    return im, title

anim = FuncAnimation(fig, update, frames=len(da.time), interval=50, blit=True)
plt.close(fig)

MOVIE_FILENAME = os.path.join(MOVIE_DIR, f"{SLICE_FIELD}_y{actual_y:.0f}.mp4")
anim.save(MOVIE_FILENAME, writer="ffmpeg", fps=20, dpi=150)
print(f"Saved: {MOVIE_FILENAME}")